# Notebook to make major changes to books/Accts/*.json 

In [1]:
import os
from pathlib import Path
import json

In [2]:
### -- ExpRev Global Change

# import json



In [3]:
import pandas as pd
from IPython.display import display, Markdown

erFN = Path.home()/ 'GDrive/Family/Assets/LLC-WBGroup/books/Accts/llcExpRev_WBGroupLLC.json'
    
def loadLedger():
    jstr = erFN.read_text()
    return json.loads(jstr)

def saveLedger(erList, erFN):
    with open(erFN,'w') as fio:
        json.dump(erList, fio)
    
def displayLedger(erList, **kwargs):
    for er in erList:
        amt = er['amt']
        pNm = er['propNm']
        aSub = er['acctSub']
        a = er['acct']
        l = er['Ledger']
    
        print(f"{pNm:13s} {amt:10.2f}: {er['aType']:7s}, {a} -> {l} ==acctSub:{aSub}")

def displayGroup(erList):
    df = pd.DataFrame(erList)
    return df.groupby(['acct', 'Ledger']).amt.sum()

def swapLedger(erList):
    for er in erList:
        amt = er['amt']
        pNm = er['propNm']
        aSub = er['acctSub']
        a = er['acct']
        l = er['Ledger']
        dc = er['aType']
        
        if er['acct'] != 'Acct.Cash.Bank' and er['Ledger'] == 'Acct.Cash.Bank':
            # swap ExpRev so Acct.Cash.banik
            er['acct'] = l
            er['Ledger'] = a
            er['aType'] = 'Credit' if dc == 'Debit' else 'Debit'
    
def cleanLedger(erOld):
    print(f"\n=========== Old ExpRev {len(erOld)} ==========")

    erList = []
    for er in erOld:
        # Clean transactions
        pNm = er['propNm']
        aSub = er['acctSub']
    
        a = er['acct']
        l = er['Ledger']
        dc = er['aType']
    

        # Ignore non RV_RV1
        if 'RV_RV1' != er['propNm']:                
            erList.append(er)
            continue
            
        # Ignore investment, purchase
        amt = er['amt']
        if amt == 177.00 :
            erList.append(er)
            #print(er)
            continue

        er['acct'] = 'Acct.Cash.Bank'
        er['Ledger'] = 'Acct.Fixed.Tangible.InConstruction'
        er['aType'] = 'Credit'
    
        print(f"{pNm:7s} {amt:10.2f}: {er['aType']:7s}, {a} -> {l} ==acctSub:{aSub}")
        erList.append(er)
    return erList

In [4]:
erOrig_List = loadLedger()
erOld_List = loadLedger()

print("=================== Orig")
dfOrig = displayGroup(erOrig_List)
display(dfOrig)
print(dfOrig.sum())

print("=================== swap")

swapLedger(erOld_List)

dfOld = displayGroup(erOld_List)
display(dfOld)
print(dfOld.sum())

=================== Orig


acct            Ledger                            
Acct.Cash.Bank  Acct.Cash.Escrow                      213936.95
                Acct.Equity.Owner.Capital.Funds       219257.00
                Acct.Exp.Other                           550.85
                Acct.Exp.Repair                          983.38
                Acct.Exp.Util                           1006.95
                Acct.Fixed.Tangible.InConstruction       177.00
                Acct.Rev.Fees.Other                      401.06
                Acct.Rev.Rent                           4000.00
Acct.Exp.Other  Acct.Cash.Bank                            96.26
Acct.Exp.Util   Acct.Cash.Bank                           181.80
Name: amt, dtype: float64

440591.25
=================== swap


acct            Ledger                            
Acct.Cash.Bank  Acct.Cash.Escrow                      213936.95
                Acct.Equity.Owner.Capital.Funds       219257.00
                Acct.Exp.Other                           647.11
                Acct.Exp.Repair                          983.38
                Acct.Exp.Util                           1188.75
                Acct.Fixed.Tangible.InConstruction       177.00
                Acct.Rev.Fees.Other                      401.06
                Acct.Rev.Rent                           4000.00
Name: amt, dtype: float64

440591.25


In [5]:
erOld_List = loadLedger()

swapLedger(erOld_List)

erNew_List = cleanLedger(erOld_List)

dfNew = displayGroup(erNew_List)  
display(dfNew)
print(dfNew.sum())

#print(f"\n=========== NEW ExpRev {len(erNew_List)} ==========")
#print(json.dumps(erNew_List, indent=4))

saveLedger(erNew_List, erFN)
        


=========== Old ExpRev 54 ==========
RV_RV1       27.04: Credit , Acct.Cash.Bank -> Acct.Exp.Other ==acctSub:None
RV_RV1       27.04: Credit , Acct.Cash.Bank -> Acct.Exp.Other ==acctSub:None
RV_RV1       31.86: Credit , Acct.Cash.Bank -> Acct.Exp.Other ==acctSub:Exp Other
RV_RV1       34.63: Credit , Acct.Cash.Bank -> Acct.Exp.Other ==acctSub:Exp Other
RV_RV1       14.06: Credit , Acct.Cash.Bank -> Acct.Exp.Other ==acctSub:None
RV_RV1       14.06: Credit , Acct.Cash.Bank -> Acct.Exp.Other ==acctSub:None
RV_RV1       37.87: Credit , Acct.Cash.Bank -> Acct.Exp.Other ==acctSub:Const
RV_RV1      108.32: Credit , Acct.Cash.Bank -> Acct.Exp.Other ==acctSub:Exp Other
RV_RV1       19.47: Credit , Acct.Cash.Bank -> Acct.Exp.Repair ==acctSub:Const
RV_RV1       30.71: Credit , Acct.Cash.Bank -> Acct.Exp.Repair ==acctSub:Const
RV_RV1       60.58: Credit , Acct.Cash.Bank -> Acct.Exp.Repair ==acctSub:Const
RV_RV1       57.31: Credit , Acct.Cash.Bank -> Acct.Exp.Repair ==acctSub:Const
RV_RV1       3

acct            Ledger                            
Acct.Cash.Bank  Acct.Cash.Escrow                      213936.95
                Acct.Equity.Owner.Capital.Funds       219257.00
                Acct.Exp.Other                           320.23
                Acct.Exp.Repair                          199.06
                Acct.Exp.Util                           1188.75
                Acct.Fixed.Tangible.InConstruction      1288.20
                Acct.Rev.Fees.Other                      401.06
                Acct.Rev.Rent                           4000.00
Name: amt, dtype: float64

440591.25


## Post Save

In [6]:
erList = loadLedger()

df = displayGroup(erList)  
display(df)
print(df.sum())


acct            Ledger                            
Acct.Cash.Bank  Acct.Cash.Escrow                      213936.95
                Acct.Equity.Owner.Capital.Funds       219257.00
                Acct.Exp.Other                           320.23
                Acct.Exp.Repair                          199.06
                Acct.Exp.Util                           1188.75
                Acct.Fixed.Tangible.InConstruction      1288.20
                Acct.Rev.Fees.Other                      401.06
                Acct.Rev.Rent                           4000.00
Name: amt, dtype: float64

440591.25
